# Módulo 09 — Redes ART (Adaptive Resonance Theory)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Desenvolvidas por Stephen Grossberg e Gail Carpenter, as redes **ART** enfrentam o dilema central do aprendizado neural:

- **Plasticidade:** capacidade de assimilar novos padrões
- **Estabilidade:** capacidade de não destruir o que foi aprendido anteriormente

Redes convencionais sofrem de **catastrophic forgetting**: aprender uma nova tarefa apaga padrões anteriores. A ART resolve isso com um **parâmetro de vigilância $\rho$** que controla quando criar uma nova categoria versus atualizar uma existente.

**Critério de vigilância (ART-1 — entradas binárias):**

$$\frac{|\mathbf{x} \cap \mathbf{w}|}{|\mathbf{x}|} \geq \rho$$

Se a similaridade entre a entrada $\mathbf{x}$ e o protótipo da categoria $\mathbf{w}$ atingir o limiar $\rho$, ocorre **ressonância**: a categoria é refinada pela interseção. Caso contrário, uma nova categoria é criada.

**Efeito de $\rho$:** valores altos criam categorias muito específicas (muitas classes finas); valores baixos agrupam padrões diferentes numa mesma categoria (menos classes grosseiras).

In [1]:
# Passo 5 — Classe ART-1 completa
import numpy as np

class ART1:
    def __init__(self, input_size, vigilance=0.7):
        self.input_size = input_size
        self.vigilance = vigilance
        self.categories = []
        self.num_categories = 0

    def compute_similarity(self, pattern, weights):
        intersection = np.sum(pattern * weights)
        pattern_sum = np.sum(pattern)
        return intersection / pattern_sum if pattern_sum > 0 else 0

    def train(self, pattern):
        if self.num_categories == 0:
            self.categories.append(pattern.copy())
            self.num_categories = 1
            return (0, False)

        tried_categories = set()
        while len(tried_categories) < self.num_categories:
            for i in range(self.num_categories):
                if i not in tried_categories:
                    category = i
                    tried_categories.add(i)
                    break
            similarity = self.compute_similarity(pattern, self.categories[category])
            if similarity >= self.vigilance:
                self.categories[category] = pattern * self.categories[category]
                return (category, False)

        self.categories.append(pattern.copy())
        self.num_categories += 1
        return (self.num_categories - 1, True)

    def predict(self, pattern):
        if self.num_categories == 0:
            return None
        best_cat = 0
        best_sim = self.compute_similarity(pattern, self.categories[0])
        for i in range(1, self.num_categories):
            sim = self.compute_similarity(pattern, self.categories[i])
            if sim > best_sim:
                best_sim = sim
                best_cat = i
        return best_cat if best_sim >= self.vigilance else None

art = ART1(input_size=6, vigilance=0.6)
print('APRENDIZADO INCREMENTAL COM ART-1')
print('='*40)

data_stream = [
    np.array([1,1,1,0,0,0]),
    np.array([1,1,0,0,0,0]),
    np.array([1,1,1,1,0,0]),
    np.array([0,0,0,1,1,1]),
    np.array([0,0,0,1,1,0]),
    np.array([1,0,0,0,0,0]),
]

for i, pattern in enumerate(data_stream):
    category, was_new = art.train(pattern)
    novo_str = ' [NOVA]' if was_new or i == 0 else ''
    print(f'Amostra {i+1}: {pattern} → Categoria {category}{novo_str}')

print(f'\nTotal de categorias: {art.num_categories}')
print('Rede aprendeu incrementalmente sem esquecer padrões anteriores!')

APRENDIZADO INCREMENTAL COM ART-1
Amostra 1: [1 1 1 0 0 0] → Categoria 0 [NOVA]
Amostra 2: [1 1 0 0 0 0] → Categoria 0
Amostra 3: [1 1 1 1 0 0] → Categoria 1 [NOVA]
Amostra 4: [0 0 0 1 1 1] → Categoria 2 [NOVA]
Amostra 5: [0 0 0 1 1 0] → Categoria 2
Amostra 6: [1 0 0 0 0 0] → Categoria 0

Total de categorias: 3
Rede aprendeu incrementalmente sem esquecer padrões anteriores!


In [2]:
# Passo 6 — Clustering de dados streaming com 3 clusters naturais
import numpy as np

np.random.seed(42)
art2 = ART1(input_size=4, vigilance=0.75)

stream = []
# Cluster A: [1,1,0,0] com variações
for _ in range(5):
    pattern = np.array([1,1,0,0])
    if np.random.rand() > 0.7:
        pattern = pattern.copy()
        pattern[np.random.randint(0,4)] = 1
    stream.append(pattern)

# Cluster B: [0,0,1,1] com variações
for _ in range(5):
    pattern = np.array([0,0,1,1])
    if np.random.rand() > 0.7:
        pattern = pattern.copy()
        pattern[np.random.randint(0,4)] = 1
    stream.append(pattern)

# Cluster C: [1,0,1,0] com variações
for _ in range(5):
    pattern = np.array([1,0,1,0])
    if np.random.rand() > 0.7:
        pattern = pattern.copy()
        pattern[np.random.randint(0,4)] = 1
    stream.append(pattern)

for i, pattern in enumerate(stream):
    category, was_new = art2.train(pattern)
    novo_str = ' [NOVA CATEGORIA]' if was_new or i == 0 else ''
    print(f'Stream {i+1:2d}: {pattern} → Cat {category}{novo_str}')

print(f'\nCategorias descobertas: {art2.num_categories}')
print('\nAgrupamentos finais:')
for i in range(art2.num_categories):
    print(f'  Categoria {i}: {art2.categories[i]}')

Stream  1: [1 1 0 0] → Cat 0 [NOVA CATEGORIA]
Stream  2: [1 1 1 0] → Cat 1 [NOVA CATEGORIA]
Stream  3: [1 1 0 0] → Cat 0
Stream  4: [1 1 0 0] → Cat 0
Stream  5: [1 1 0 0] → Cat 0
Stream  6: [0 0 1 1] → Cat 2 [NOVA CATEGORIA]
Stream  7: [0 0 1 1] → Cat 2
Stream  8: [0 0 1 1] → Cat 2
Stream  9: [0 0 1 1] → Cat 2
Stream 10: [0 0 1 1] → Cat 2
Stream 11: [1 1 1 0] → Cat 1
Stream 12: [1 0 1 0] → Cat 1
Stream 13: [1 0 1 0] → Cat 1
Stream 14: [1 0 1 0] → Cat 1
Stream 15: [1 0 1 0] → Cat 1

Categorias descobertas: 3

Agrupamentos finais:
  Categoria 0: [1 1 0 0]
  Categoria 1: [1 0 1 0]
  Categoria 2: [0 0 1 1]


## Análise Crítica

**Solução ao catastrophic forgetting:** A ART-1 é a única arquitetura deste portfólio que genuinamente preserva padrões anteriores ao assimilar novos. A criação de novas categorias protege as memórias existentes — um padrão suficientemente diferente nunca sobrescreve uma categoria já formada.

**Papel da vigilância $\rho$:** valores altos (ex: 0.9) geram muitas categorias finas — cada variação pequena cria uma nova classe (análogo ao overfitting). Valores baixos (ex: 0.3) agrupam padrões muito diferentes numa mesma categoria (análogo ao underfitting).

**Ordem de apresentação importa:** diferente do SOM que processa todos os dados em cada época, a ART-1 aprende de forma estritamente incremental. A ordem em que os padrões chegam influencia quais categorias são formadas e como são refinadas.

**Limitação de escalabilidade:** a ART-1 trabalha apenas com entradas binárias. Extensões como ART-2 e Fuzzy ART lidam com entradas contínuas, mas aumentam a complexidade do mecanismo de vigilância.